In [ ]:
!pip install -q timm torchaudio kagglehub

In [ ]:
import os
import kagglehub

os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")

path = kagglehub.competition_download('itmo-acoustic-event-detectin-2026')
print("Path:", path)

competition_path = path

train_folder = os.path.join(competition_path, "audio_train", "train")
train_csv = os.path.join(competition_path, "train.csv")

print("Files:", len(os.listdir(train_folder)))

Path: /root/.cache/kagglehub/competitions/itmo-acoustic-event-detectin-2026
Files: 5683


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import timm

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

In [14]:
TARGET_SR = 32000
MAX_LEN = TARGET_SR * 3

BATCH_SIZE = 32
EPOCHS = 15
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
df = pd.read_csv(train_csv, skiprows=1, names=["fname", "label"])
df["path"] = df["fname"].apply(lambda x: os.path.join(train_folder, x))

labels = sorted(df["label"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

df["label_id"] = df["label"].map(label2id)

In [16]:
def load_audio(path):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim=0)

    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)

    if len(wav) <= MAX_LEN:
        wav = torch.nn.functional.pad(wav, (0, MAX_LEN - len(wav)))
    else:
        max_start = len(wav) - MAX_LEN
        start = np.random.randint(0, max_start) if max_start > 0 else 0
        wav = wav[start:start + MAX_LEN]

    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=TARGET_SR,
        n_fft=1024,
        hop_length=320,
        n_mels=128
    )(wav)

    mel = torch.log(mel + 1e-6)

    delta = torchaudio.functional.compute_deltas(mel)
    delta2 = torchaudio.functional.compute_deltas(delta)

    return torch.stack([mel, delta, delta2])

In [17]:
class AudioDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = load_audio(row["path"])

        if self.augment:
            if np.random.rand() < 0.5:
                x = x + 0.01 * torch.randn_like(x)

        return x, row["label_id"]

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

train_df, val_df = train_test_split(df, test_size=0.1, stratify=df["label_id"], random_state=SEED)

train_dataset = AudioDataset(train_df, augment=True)
val_dataset = AudioDataset(val_df, augment=False)

class_counts = np.bincount(train_df["label_id"])
weights = 1. / class_counts
sample_weights = weights[train_df["label_id"]]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [20]:
model = timm.create_model(
    "tf_efficientnetv2_s",
    pretrained=True,
    num_classes=len(label2id),
    in_chans=3
).to(DEVICE)

In [21]:
def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0)).to(DEVICE)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    all_preds, all_targets = [], []

    for x, y in tqdm(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)

        x, y_a, y_b, lam = mixup(x, y)

        optimizer.zero_grad()
        out = model(x)

        loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
        loss.backward()
        optimizer.step()

        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    train_f1 = f1_score(all_targets, all_preds, average="weighted")

    # validation
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)

            preds = out.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    val_f1 = f1_score(all_targets, all_preds, average="weighted")

    print(f"Epoch {epoch+1} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pt")

100%|██████████| 160/160 [01:46<00:00,  1.50it/s]


Epoch 1 | Train F1: 0.2231 | Val F1: 0.5711


100%|██████████| 160/160 [01:45<00:00,  1.52it/s]


Epoch 2 | Train F1: 0.3442 | Val F1: 0.6523


100%|██████████| 160/160 [01:46<00:00,  1.50it/s]


...

Epoch 10 | Train F1: 0.4868 | Val F1: 0.7603


100%|██████████| 160/160 [01:47<00:00,  1.49it/s]


Epoch 11 | Train F1: 0.4675 | Val F1: 0.7916


100%|██████████| 160/160 [01:45<00:00,  1.51it/s]


Epoch 12 | Train F1: 0.4987 | Val F1: 0.7941


In [22]:
def predict_tta(path, n=5):
    preds = []

    for _ in range(n):
        x = load_audio(path).unsqueeze(0).to(DEVICE)
        out = torch.softmax(model(x), dim=1)
        preds.append(out)

    return torch.stack(preds).mean(0).argmax(1).item()

In [23]:
# Пути к тестовым данным
competition_path = "/root/.cache/kagglehub/competitions/itmo-acoustic-event-detectin-2026"
test_folder = os.path.join(competition_path, "audio_test", "test")

if not os.path.exists(test_folder):
    test_folder = os.path.join(competition_path, "audio_test")

print(f"Test folder: {test_folder}")
test_files = sorted(os.listdir(test_folder))
print(f"Test files: {len(test_files)}")

Test folder: /root/.cache/kagglehub/competitions/itmo-acoustic-event-detectin-2026/audio_test/test
Test files: 3790


In [24]:
test_files = sorted(os.listdir(test_folder))

preds = []
filenames = []

model.load_state_dict(torch.load("best_model.pt"))
model.eval()

for fname in tqdm(test_files):
    path = os.path.join(test_folder, fname)

    pred = predict_tta(path, n=5)

    preds.append(pred)
    filenames.append(fname)

pred_labels = [id2label[p] for p in preds]

submission = pd.DataFrame({
    "fname": filenames,
    "label": pred_labels
})

submission.to_csv("submission.csv", index=False)

100%|██████████| 3790/3790 [12:22<00:00,  5.10it/s]


## Вторая модель

In [ ]:
model2 = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=len(label2id),
    in_chans=3
).to(DEVICE)

In [ ]:
EPOCHS = 6
LR = 1e-3

In [30]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model2.parameters(), lr=1e-3)

best_f1 = 0

for epoch in range(EPOCHS):
    model2.train()
    all_preds, all_targets = [], []

    for x, y in tqdm(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)

        x, y_a, y_b, lam = mixup(x, y)

        optimizer.zero_grad()
        out = model2(x)

        loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
        loss.backward()
        optimizer.step()

        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.cpu().numpy())

    train_f1 = f1_score(all_targets, all_preds, average="weighted")

    # validation
    model2.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model2(x)

            preds = out.argmax(1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    val_f1 = f1_score(all_targets, all_preds, average="weighted")

    print(f"Epoch {epoch+1} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model2.state_dict(), "model_b0.pt")

100%|██████████| 160/160 [01:26<00:00,  1.85it/s]


Epoch 1 | Train F1: 0.2300 | Val F1: 0.5837


100%|██████████| 160/160 [01:23<00:00,  1.92it/s]


Epoch 2 | Train F1: 0.3687 | Val F1: 0.6324


100%|██████████| 160/160 [01:20<00:00,  1.99it/s]


...

Epoch 4 | Train F1: 0.4525 | Val F1: 0.6917


100%|██████████| 160/160 [01:22<00:00,  1.95it/s]


Epoch 5 | Train F1: 0.3637 | Val F1: 0.7000


100%|██████████| 160/160 [01:21<00:00,  1.96it/s]


Epoch 6 | Train F1: 0.4229 | Val F1: 0.7581


In [ ]:
# модель 1
model1 = timm.create_model(
    "tf_efficientnetv2_s",
    pretrained=False,
    num_classes=len(label2id),
    in_chans=3
).to(DEVICE)

model1.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
model1.eval()

# модель 2
model2 = timm.create_model(
    "efficientnet_b0",
    pretrained=False,
    num_classes=len(label2id),
    in_chans=3
).to(DEVICE)

model2.load_state_dict(torch.load("model_b0.pt", map_location=DEVICE))
model2.eval()

In [ ]:
def predict_ensemble_tta(path, n=5):
    preds = []

    for _ in range(n):
        x = load_audio(path).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            out1 = torch.softmax(model1(x), dim=1)
            out2 = torch.softmax(model2(x), dim=1)

            out = 0.65 * out1 + 0.35 * out2

        preds.append(out)

    return torch.stack(preds).mean(0).argmax(1).item()

In [33]:
preds = []
filenames = []

for fname in tqdm(test_files):
    path = os.path.join(test_folder, fname)

    pred = predict_ensemble_tta(path, n=5)

    preds.append(pred)
    filenames.append(fname)

100%|██████████| 3790/3790 [13:00<00:00,  4.85it/s]


In [ ]:
pred_labels = [id2label[p] for p in preds]

submission = pd.DataFrame({
    "fname": filenames,
    "label": pred_labels
})

submission.to_csv("ensemble_submission.csv", index=False)